In [1]:
from pathlib import Path
import pandas as pd
from rdflib import Graph, URIRef
from rdflib.namespace import OWL, RDFS, XSD

# Start from the notebook's current directory
NB_DIR = Path.cwd().resolve()

# Walk up to find a directory that contains a 'src' folder
SRC_DIR = None
for p in [NB_DIR] + list(NB_DIR.parents):
    candidate = p / "assignment" / "src"
    if candidate.exists():
        SRC_DIR = candidate
        break
    candidate = p / "src"  # also allow projects that don't have the extra 'assignment' level
    if candidate.exists():
        SRC_DIR = candidate
        break

if SRC_DIR is None:
    raise FileNotFoundError(
        f"Could not find a 'src' directory above {NB_DIR}. "
        "If your project root is at e.g. .../assignment/, open the notebook there or set SRC_DIR manually."
    )

DATA_DIR = SRC_DIR / "data"

print("Notebook CWD:   ", NB_DIR)
print("Resolved SRC_DIR:", SRC_DIR)
print("Resolved DATA_DIR:", DATA_DIR)
print("TTL files found:", [p.name for p in SRC_DIR.glob("*.ttl")])


TTL = {
    "bfo":  SRC_DIR / "bfo-core.ttl",
    "ies":  SRC_DIR / "ies.ttl",
    "qudt": SRC_DIR / "qudt.ttl",
    "ccom": SRC_DIR / "ccom.ttl",
    "ccot": SRC_DIR / "ccot.ttl",
    "to":   SRC_DIR / "time.ttl",
}

OUT = {
    key: DATA_DIR / f"{key}-axioms.xlsx"
    for key in TTL
}

CLASS_AXIOM_QUERY = """
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?iri ?label ?predicate ?object
WHERE {
  ?iri a ?classType .
  VALUES ?classType { owl:Class rdfs:Class }

  OPTIONAL { ?iri rdfs:label ?label }

  FILTER NOT EXISTS {
    ?iri owl:deprecated ?d .
    FILTER(?d = true || lcase(str(?d)) = "true")
  }

  FILTER(!isBlank(?iri))

  ?iri ?predicate ?object .
  FILTER(
    ?predicate IN (
      rdfs:subClassOf,
      owl:equivalentClass,
      owl:disjointWith,
      owl:disjointUnionOf
    )
    || (?predicate = rdf:type && ?object = owl:Class)
  )
}
ORDER BY ?label ?iri ?predicate
"""

from rdflib import BNode

def run_local_select(ttl_path: Path, query: str) -> pd.DataFrame:
    g = Graph()
    g.parse(ttl_path, format="turtle")

    def axiom_text(iri, predicate, obj):
        triples = [(iri, predicate, obj)]
        visited = set()

        def expand(node):
            if not isinstance(node, BNode) or node in visited:
                return

            visited.add(node)
            for p, o in g.predicate_objects(node):
                triples.append((node, p, o))
                expand(o)

        expand(obj)

        return "\n".join(
            f"{s.n3(g.namespace_manager)} "
            f"{p.n3(g.namespace_manager)} "
            f"{o.n3(g.namespace_manager)} ."
            for s, p, o in triples
        )

    rows = []
    for row in g.query(query):
        rows.append({
            "iri": str(row.iri),
            "label": str(row.label) if row.label is not None else None,
            "axiom": axiom_text(row.iri, row.predicate, row.object),
        })

    return pd.DataFrame(rows, columns=["iri", "label", "axiom"])


DATA_DIR.mkdir(parents=True, exist_ok=True)

for key, ttl_path in TTL.items():
    if not ttl_path.exists():
        print(f"Missing {ttl_path} — skipping {key}")
        continue

    df = run_local_select(ttl_path, CLASS_AXIOM_QUERY)
    df.to_excel(OUT[key], index=False)
    print(f"Saved {len(df)} rows to {OUT[key]}")



Notebook CWD:    C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\notebooks
Resolved SRC_DIR: C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src
Resolved DATA_DIR: C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src\data
TTL files found: ['bfo-core.ttl', 'ccom.ttl', 'ccot.ttl', 'ies.ttl', 'qudt.ttl', 'time.ttl']
Saved 107 rows to C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src\data\bfo-axioms.xlsx
Saved 576 rows to C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src\data\ies-axioms.xlsx
Saved 401 rows to C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src\data\qudt-axioms.xlsx
Saved 55 rows to C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src\data\ccom-axioms.xlsx
Saved 82 rows to C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src\data\ccot-axioms.xlsx
Saved 154 rows to C:\Users\kygal\repos\Ontolog